In [8]:
import json
import warnings
import google.generativeai as genai

warnings.filterwarnings("ignore")

In [ ]:
API_KEY = ""
genai.configure(api_key=API_KEY)

model_name = 'gemini-3.5-flash-lite'
model = genai.GenerativeModel(model_name)

sample_text = """
Artificial Intelligence (AI) is the ability of computer systems to perform tasks that typically require human intelligence. 
Machine learning is a subset of AI that allows systems to learn from data without direct programming. 
Deep learning, in turn, uses multi-layered neural networks to analyze complex patterns.
Despite the progress, modern AI is "narrow" (Weak AI), meaning it only solves specific tasks. 
The creation of Artificial General Intelligence (AGI), which would surpass humans in all areas, remains an open question and carries ethical risks.
"""
print("✅ Environment is set up, model is connected, text is loaded!")

✅ Environment is set up, model is connected, text is loaded!


In [3]:
print("Task 1: simple promt and advanced one.\n")

prompt_simple = f"Analyze the text and determine its main ideas:\n{sample_text}"
response_simple = model.generate_content(prompt_simple)
print("Simple promt\n", response_simple.text, "\n")

prompt_advanced = f"""
You are a university professor teaching an AI course. Analyze the provided text.
Complete the task strictly step by step:
1. Determine the main theme (1 sentence).
2. Highlight 3 key ideas (as a bulleted list).
3. Formulate a brief conclusion for students.

Text:
{sample_text}
"""
response_advanced = model.generate_content(prompt_advanced)
print("Advanced promt.\n", response_advanced.text)

Task 1: simple promt and advanced one.

Simple promt
 Based on the provided text, the main ideas can be broken down as follows:

1. **Definition of AI:** Artificial Intelligence enables computer systems to perform tasks that normally require human intelligence.
2. **Hierarchy of AI Technologies:** AI encompasses machine learning (learning from data without explicit programming), which further encompasses deep learning (using multi-layered neural networks to recognize complex patterns).
3. **Current Limitations:** Modern AI is strictly "narrow" or "weak," meaning it is limited to solving specific, individual tasks rather than possessing broad intelligence.
4. **Future Outlook and Risks:** The potential development of Artificial General Intelligence (AGI)—which could outperform humans across all areas—remains an unresolved scientific question and presents significant ethical risks. 

Advanced promt.
 Welcome to class, everyone. Let's break down the provided text according to your instruc

In [ ]:
print("Task 3: Reaction imitation\n")

prompt_react = f"""
Analyze the text. Determine what is a fact and what is an opinion/prediction.
Work STRICTLY according to the following template:
ACTION PLAN: [describe how you will approach this]
EXECUTION: [conduct the analysis]
VERIFICATION: [check your results]
RESULT: [final conclusion]

Text: {sample_text}
"""
response_react = model.generate_content(prompt_react)
print(response_react.text)

Task 3: React imitation

ACTION PLAN: [Read the provided text carefully, sentence by sentence, and classify each statement as either a "Fact" (verifiable, objective information, established definitions, or current consensus) or an "Opinion/Prediction" (subjective views, value judgments, or projections about the future). Format the output strictly according to the requested template.]

EXECUTION: 
1. "Artificial Intelligence (AI) is the ability of computer systems to perform tasks that typically require human intelligence." -> Fact (Standard technical definition of AI).
2. "Machine learning is a subset of AI that allows systems to learn from data without direct programming." -> Fact (Standard technical definition of machine learning).
3. "Deep learning, in turn, uses multi-layered neural networks to analyze complex patterns." -> Fact (Standard technical definition of deep learning).
4. "Despite the progress, modern AI is 'narrow' (Weak AI), meaning it only solves specific tasks." -> Fac

In [5]:
print("Task 4: structured(JSON).\n")

json_model = genai.GenerativeModel(model_name, generation_config={"response_mime_type": "application/json"})

prompt_json = f"""
Analyze the text. Return the result ONLY in JSON format using this schema:
{{
  "topic": "topic",
  "keywords": ["word1", "word2"],
  "conclusion": "conclusion"
}}
Text: {sample_text}
"""
response_json = json_model.generate_content(prompt_json)

try:
    parsed_data = json.loads(response_json.text)
    print("✅ Successfully parsed JSON!")
    print(f"Keywords (as Python List): {parsed_data.get('keywords')}")
    print("\nFull JSON:")
    print(json.dumps(parsed_data, indent=2, ensure_ascii=False))
except json.JSONDecodeError:
    print("Error: The model returned invalid JSON")

Task 4: structured(JSON).

✅ Successfully parsed JSON!
Keywords (as Python List): ['Artificial Intelligence', 'Machine learning', 'Deep learning', 'Weak AI', 'Artificial General Intelligence']

Full JSON:
{
  "topic": "Artificial Intelligence",
  "keywords": [
    "Artificial Intelligence",
    "Machine learning",
    "Deep learning",
    "Weak AI",
    "Artificial General Intelligence"
  ],
  "conclusion": "Modern AI is currently limited to narrow tasks, while the future development of Artificial General Intelligence (AGI) remains an open and ethically risky question."
}


In [6]:
print("Task 5: working with big context.\n")

chunks = [
    "Part 1: Computers appeared in the mid-20th century. Initially, they occupied entire rooms and performed simple mathematical calculations.",
    "Part 2: The 1980s marked the beginning of the personal computer era. People gained access to PCs at home, and the first mass operating systems emerged.",
    "Part 3: The 21st century brought a boom in the internet and mobile technologies. Smartphones became more powerful than early computers, and cloud technologies appeared."
]

full_text = " ".join(chunks)
prompt_no_chunk = f"Make a brief 1-sentence summary of the text:\n{full_text}"
print("--- 5.1 Without Chunking ---\n", model.generate_content(prompt_no_chunk).text, "\n")

print("--- 5.2 With Chunking (Step-by-step) ---")
chunk_summaries = []

for i, chunk in enumerate(chunks):
    prompt_chunk = f"Write the main idea of this excerpt in 1 sentence:\n{chunk}"
    summary = model.generate_content(prompt_chunk).text.strip()
    chunk_summaries.append(summary)
    print(f"Summary for Chunk {i+1}: {summary}")

joined_summaries = "\n".join(chunk_summaries)
prompt_final_reduce = f"Here are the summaries from different parts of the document. Combine them into one final overall text:\n{joined_summaries}"
print("\n--- Final Chunking Result ---\n", model.generate_content(prompt_final_reduce).text)

Task 5: working with big context.

--- 5.1 Without Chunking ---
 Computers have evolved rapidly since the mid-20th century, transitioning from massive room-sized machines to accessible home personal computers, and finally to today's powerful mobile and internet-connected devices. 

--- 5.2 With Chunking (Step-by-step) ---
Summary for Chunk 1: When computers first appeared in the mid-20th century, they were massive machines that filled entire rooms and were used only for basic mathematical calculations.
Summary for Chunk 2: The 1980s launched the personal computer era by introducing home PCs and the first mass operating systems.
Summary for Chunk 3: The 21st century experienced a major technological boom characterized by the rise of powerful smartphones, mobile technologies, and cloud computing
.

--- Final Chunking Result ---
 Here is the combined overall text:

When computers first appeared in the mid-20th century, they were massive machines that filled entire rooms and were used only

In [7]:
print("Task 6: temperature and top.\n")

creative_prompt = "Come up with an original and funny name for a company that develops smart robot cats."

print("--- Temperature Exploration ---")
for temp in [0.0, 0.5, 1.0]:
    config_temp = genai.types.GenerationConfig(temperature=temp)
    res = model.generate_content(creative_prompt, generation_config=config_temp)
    print(f"Temperature {temp}: {res.text.strip()}")

print("\n--- Top-P Exploration (at Temp=0.8) ---")
for p in [0.5, 0.8, 1.0]:
    config_p = genai.types.GenerationConfig(temperature=0.8, top_p=p)
    res = model.generate_content(creative_prompt, generation_config=config_p)
    print(f"Top-P {p}: {res.text.strip()}")

Task 6: temperature and top.

--- Temperature Exploration ---
Temperature 0.0: **PurrProcessor Inc.** 
*(Tagline: "Zero litter. Infinite compute.")*

**Other honorable mentions depending on the vibe:**

*   **Meowtrons:** For when you want a pet that purrs, judges you silently, and runs on a lithium-ion battery.
*   **Schrödinger’s Pet:** Is it alive? Is it a machine? Yes, and it still ignores you when you call its name. 
*   **Caterpillar (Cat-O-Pillar):** For the robot cat that undergoes a terrifying "software update" instead of a cocoon phase. 
*   **Pawgrammers:** For cats that don't just chase laser pointers; they write the code for them.
Temperature 0.5: **C
yborg Meow-Nufacturing** 

*(Tagline: "All the aloof judgment of a real cat, now with Wi-Fi capability.")*

**Other honorable mentions:**
*   **Purr-cessor Technologies** (For the tech-heads)
*   **Meowtrice** (Because you’re living in a simulation, and it’s controlled by a robotic cat)
*   **Hiss & Microchip**
Temperature 1.